# Some canopy transmissivity modeling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_palette("tab20")

import sys
import pandas as pd
import xarray as xr
from pathlib import PurePath

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as hel

In [ ]:
def diffuse_radiation_subcanopy(S_toc, tau):
    """
    Calculate the diffuse radiation under a canopy.

    Parameters:
    S_toc (float): Total incoming diffuse shortwave radiation at the top of the canopy (W/m^2).
    tau (float): Transmissivity of the canopy (dimensionless).

    Returns:
    float: Diffuse radiation under the canopy (W/m^2).
    """

    # Shortwave diffuse radiation S_d
    S_d = S_toc * tau
    return S_d

def direct_radiation_subcanopy(S_toc, mu, h, theta):
    """
    Calculate the direct radiation under a canopy.

    Parameters:
    S_toc (float): Total incoming shortwave radiation at the top of the canopy (W/m^2).
    mu (float): Extinction coefficient of the canopy [m-1].
    h (float): Canopy height [m].
    theta (float): Solar zenith angle (degrees).

    Returns:
    float: Direct radiation under the canopy (W/m^2).
    """

    # Shortwave beam radiation S_b
    S_b = S_toc * np.exp(-mu * h / np.cos(np.radians(theta)))
    return S_b

In [ ]:
# Optical transmissivity values for a canopy (how much light is passing through) from Table 1 (Link and Marks, 1999)
tau_values = np.array([1, 0.44, 0.30, 0.20, 0.16])

# Calculate and plot diffuse radiation under the canopy
plt.figure(figsize=(6, 4))
S_toc = 200  # Example value for total incoming diffuse radiation (W/m^2)
for tau in tau_values:
    S_d = diffuse_radiation_subcanopy(S_toc, tau)
    plt.plot(tau, S_d, 'o', label=f'Tau = {tau:.1f}')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('Transmissivity (Tau)')
plt.ylabel('Diffuse Radiation Under Canopy (W/m^2)')
plt.title('Diffuse Radiation Under Canopy vs Transmissivity')

In [ ]:
# Extinction coefficients from Table 1 of Link and Marks (1999)
mu_list = np.array([0, 0.025, 0.033, 0.040, 0.074])
# These seem really low...
print("Extinction coefficients (mu):", mu_list)

# Canopy height in LANDFIRE data
h_list = np.array([0, .25, .75, 1., 2., 2.5, 3., 7.5, 17.5, 37.5])
print("Canopy heights (h):", h_list)

# Solar zenith angle in degrees
# This angle varies based on latitude, time of year and time of day
# At noon, this angle is at a minimum (0 degrees) when the sun is directly overhead and is parallel to the zenith
# At sunrise and sunset, the angle is at a maximum (90 degrees) when the sun is on the horizon
# On average in mid-latitude regions, this angle reaches a minimum in the summerimte and maximum in the wintertime
# Let's keep this simple for demonstration purposes since we are modifying canopy, not solar position
theta = np.array([45])
print("Solar zenith angles (theta):", theta)

# Total incoming shortwave radiation at the top of the canopy
S_toc = 1000  # W/m^2, example value for total incoming shortwave radiation

# Calculate the direct radiation under the canopy for each combination of mu, h, and theta without using loops
S_b_matrix = np.zeros((len(mu_list), len(h_list)))
for i, mu in enumerate(mu_list):
    for j, h in enumerate(h_list):
            S_b_matrix[i, j] = direct_radiation_subcanopy(S_toc, mu, h, theta)


# Plot the results for each combination of mu and h
plt.figure(figsize=(10, 6))
markers = ['o', 's', 'D', '^', 'v', '.', '_', 'd', 'o']  # Different markers for each extinction coefficient
for i, mu in enumerate(mu_list):
    plt.plot(h_list, S_b_matrix[i], marker=markers[i], label=f'Extinction Coefficient = {mu:.2f}')
plt.title('Direct Radiation Under Canopy vs. Canopy Height')
plt.xlabel('Canopy Height (m)')
plt.ylabel('Direct Radiation (W/m^2)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
# Extinction coefficients from Table 1 of Link and Marks (1999)
mu_list = np.array([0, 0.025, 0.033, 0.040, 0.074])
# These seem really low...
print("Extinction coefficients (mu):", mu_list)

# Abbreviated canopy height list in LANDFIRE data
h_list = np.array([0, .25, .75, 1., 2., 2.5, 3.])
print("Canopy heights (h):", h_list)

# Solar zenith angle in degrees
# This angle varies based on latitude, time of year and time of day
# At noon, this angle is at a minimum (0 degrees) when the sun is directly overhead and is parallel to the zenith
# At sunrise and sunset, the angle is at a maximum (90 degrees) when the sun is on the horizon
# On average in mid-latitude regions, this angle reaches a minimum in the summerimte and maximum in the wintertime
# Let's keep this simple for demonstration purposes since we are modifying canopy, not solar position
theta = np.array([45])
print("Solar zenith angles (theta):", theta)

# Total incoming shortwave radiation at the top of the canopy
S_toc = 1000  # W/m^2, example value for total incoming shortwave radiation

# Calculate the direct radiation under the canopy for each combination of mu, h, and theta without using loops
S_b_matrix = np.zeros((len(mu_list), len(h_list)))
for i, mu in enumerate(mu_list):
    for j, h in enumerate(h_list):
            S_b_matrix[i, j] = direct_radiation_subcanopy(S_toc, mu, h, theta)


# Plot the results for each combination of mu and h
plt.figure(figsize=(10, 6))
markers = ['o', 's', 'D', '^', 'v', '.', '_', 'd', 'o']  # Different markers for each extinction coefficient
for i, mu in enumerate(mu_list):
    plt.plot(h_list, S_b_matrix[i], marker=markers[i], label=f'Extinction Coefficient = {mu:.2f}')
plt.title('Direct Radiation Under Canopy vs. Canopy Height')
plt.xlabel('Canopy Height (m)')
plt.ylabel('Direct Radiation (W/m^2)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

| As canopy height increases, direct radiation plummets, regardless of designated extinction coefficient (mu)

| At a single height (below 7.5 m -- 30 feet or so), increasing extinction coefficient exponentially decreases direct radiation

| What is a reasonable range for extinction coefficients?


# Try a more reasonable set of parameters now

In [ ]:
# Extinction coefficients from 0 to 0.7
mu_list = np.arange(0, 0.8, 0.1)
print("Extinction coefficients (mu):", mu_list)

# Canopy height in LANDFIRE data
h_list = np.array([0, .25, .75, 1., 2., 2.5, 3., 7.5, 17.5, 37.5])
print("Canopy heights (h):", h_list)

# Total incoming shortwave radiation at the top of the canopy
S_toc = 1000  # W/m^2, example value for total incoming shortwave radiation

# Calculate the direct radiation under the canopy for each combination of mu, h, and theta without using loops
S_b_matrix = np.zeros((len(mu_list), len(h_list)))
for i, mu in enumerate(mu_list):
    for j, h in enumerate(h_list):
            S_b_matrix[i, j] = direct_radiation_subcanopy(S_toc, mu, h, theta)


# Plot the results for each combination of mu and h
plt.figure(figsize=(10, 6))
markers = ['o', 's', 'D', '^', 'v', '.', '_', 'd', 'o', '_']  # Different markers for each extinction coefficient
for i, mu in enumerate(mu_list):
    plt.plot(h_list, S_b_matrix[i], marker=markers[i], label=f'Extinction Coefficient = {mu:.1f}')
plt.title('Direct Radiation Under Canopy vs. Canopy Height')
plt.xlabel('Canopy Height (m)')
plt.ylabel('Direct Radiation (W/m^2)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')


In [ ]:
for i, h in enumerate(h_list):
    plt.plot(mu_list, S_b_matrix[:, i], marker=markers[i], label=f'Canopy height = {h:.2f}m')
plt.title('Direct Radiation Under Canopy vs. Extinction Coefficient')
plt.xlabel('Extinction Coefficient')
plt.ylabel('Direct Radiation (W/m^2)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')


In [ ]:
# Calculate the percentage reduction in direct radiation for each canopy height level based on the extinction coefficient values of 0.074 and 0.71
for h in [0, .25, .75, 1., 2., 3.]:
    links_marks = direct_radiation_subcanopy(S_toc, 0.074, h, theta)[0]
    liston_elder = direct_radiation_subcanopy(S_toc, 0.71, h, theta)[0]
    perc_diff = ((links_marks - liston_elder) / links_marks) * 100
    # print(perc_diff)
    print(f'Canopy height {h} m: {perc_diff:.0f}% reduction')


In [ ]:
# Extinction coefficients from 0 to 0.7
mu_list = np.arange(0, 0.7, 0.08)
print("Extinction coefficients (mu):", mu_list)

# More realistic canopy height values
h_list = np.array([0, .25, .5, .75, 1., 2., 3., 5., 10., 15., 20.])
print("Canopy heights (h):", h_list)

# Total incoming shortwave radiation at the top of the canopy
S_toc = 1000  # W/m^2, example value for total incoming shortwave radiation

# Calculate the direct radiation under the canopy for each combination of mu, h, and theta without using loops
S_b_matrix = np.zeros((len(mu_list), len(h_list)))
for i, mu in enumerate(mu_list):
    for j, h in enumerate(h_list):
            S_b_matrix[i, j] = direct_radiation_subcanopy(S_toc, mu, h, theta)


# Plot the results for each combination of mu and h
plt.figure(figsize=(10, 6))
markers = ['o', 's', 'D', '^', 'v', '.', '_', 'd', 'o']  # Different markers for each extinction coefficient
for i, mu in enumerate(mu_list):
    plt.plot(h_list, S_b_matrix[i], marker=markers[i], label=f'Extinction Coefficient = {mu:.2f}')
plt.title('Direct Radiation Under Canopy vs. Canopy Height')
plt.xlabel('Canopy Height (m)')
plt.ylabel('Direct Radiation (W/m^2)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')


In [ ]:
# Extinction coefficients from 0 to 0.7
mu_list = np.arange(0, 0.7, 0.08)
print("Extinction coefficients (mu):", mu_list)

# More realistic abbreviated list of canopy height values
h_list = np.array([0, .25, .5, .75, 1., 2., 3.])
print("Canopy heights (h):", h_list)

# Total incoming shortwave radiation at the top of the canopy
S_toc = 1000  # W/m^2, example value for total incoming shortwave radiation

# Calculate the direct radiation under the canopy for each combination of mu, h, and theta without using loops
S_b_matrix = np.zeros((len(mu_list), len(h_list)))
for i, mu in enumerate(mu_list):
    for j, h in enumerate(h_list):
            S_b_matrix[i, j] = direct_radiation_subcanopy(S_toc, mu, h, theta)


# Plot the results for each combination of mu and h
plt.figure(figsize=(10, 6))
markers = ['o', 's', 'D', '^', 'v', '.', '_', 'd', 'o']  # Different markers for each extinction coefficient
for i, mu in enumerate(mu_list):
    plt.plot(h_list, S_b_matrix[i], marker=markers[i], label=f'Extinction Coefficient = {mu:.2f}')
plt.title('Direct Radiation Under Canopy vs. Canopy Height')
plt.xlabel('Canopy Height (m)')
plt.ylabel('Direct Radiation (W/m^2)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
# Quick calculations for LAI and mu and tau
# LAI
# canopy_density = 0.29 * np.log(LAI) + 0.55
# tau = 1 - canopy_density
# mu = (-np.log(tau) / h)

# Link and Marks tau and LANDFIRe h from above to derive mu
def derive_extinction_coefficient(tau, h):
    mu = (-np.log(tau) / h)
    return mu

param_dict_list = []
# LANDFIRE canopy height values
lm_hlist = [.25, .75, 1., 2., 2.5, 3., 7.5, 17.5, 37.5]
# Modify for theoretical values
lm_hlist = [1, 2, 3, 5, 10, 20, 37.5]
for h in lm_hlist:
    print(f'Canopy height {h} m')
    param_dict = dict()
    for tau in tau_values:
        if tau > 0:
            mu = derive_extinction_coefficient(tau, h)
            print(f'Transmissivity {tau:.2f}: Extinction Coefficient = {mu:.3f} m^-1')
            param_dict[tau] = mu
    param_dict_list.append(param_dict)

In [ ]:
# convert to dataframe
df = pd.DataFrame(param_dict_list, index=lm_hlist)
# Transpose the dataframe for better readability so the tau values are rows and the canopy heights are columns
df = df.T
df

In [ ]:
# truncate to two decimal places
df.round(2)

In [ ]:
fig, ax = plt.subplots()
df.plot(ax=ax, marker='.', linewidth=0.5)
ax.set_xlabel('Transmittivity (Tau)')
ax.set_ylabel('Extinction Coefficient (m^-1)')
ax.legend('Canopy Height (m) = ' + df.columns.astype(str),
        #   bbox_to_anchor=(1.05, 1), loc='upper left'
          )

In [ ]:
# Read in the Blue topo.nc and get the tau and k (mu) values
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts/'
basin = 'blue'
topo_fn = hel.fn_list(script_dir, f'{basin}_setup/output_100m/topo.nc')[0]
print(topo_fn)

# Read it in
topo = xr.open_dataset(topo_fn)

# Plot the distribution of k, tau, and height
fig, axa = plt.subplots(1, 3, figsize=(15, 5))
varlist = ['veg_k', 'veg_tau', 'veg_height']
for jdx, ax in enumerate(axa.flatten()):
    da = topo[varlist[jdx]]
    da.plot.hist(ax=ax, bins=30, ec='k')
    for patch in ax.patches:
        if patch.get_height() > 100:
            # Plot the bar x-value above each bar
            ax.text(patch.get_x() + patch.get_width() / 2, patch.get_height()+3000,
                    f'{patch.get_x()+ patch.get_width() :.2f}', ha='center', va='bottom')
            # Plot the percentage of the total values above each bar
            percentage = (patch.get_height() / da.size) * 100
            ax.text(patch.get_x() + patch.get_width() / 2, patch.get_height(),
                    f'({percentage:.0f}%)', ha='center', va='bottom')
plt.suptitle(f'{basin.capitalize()} River Basin Topo veg param');

In [ ]:
# Simplified framework for impacts of incorrect mu on sub-canopy direct radiation
mu_list = [0, 0.025, 0.07, 0.1, 0.15, 0.2]
d = 5 # m, example value for canopy depth
for mu in mu_list:
    print(f'I = {np.exp(-mu * d):.2f} * I_0, mu = {mu:.2f}')

In [ ]:
# what about for varying SZA?
# Extinction coefficients from Table 1 of Link and Marks (1999)
# mu_list = np.array([0, 0.025, 0.033, 0.040, 0.074])
mu_list = np.array([0.025, 0.040, 0.074])
print("Extinction coefficients (mu):", mu_list)

# Canopy height in LANDFIRE data
# h_list = np.array([0, .25, .75, 1., 2., 2.5, 3., 7.5, 17.5, 37.5])
h_list = np.array([1., 2., 7.5, 17.5, 37.5])
print("Canopy heights (h):", h_list)

# Solar zenith angle in degrees
# This angle varies based on latitude, time of year and time of day
# At noon, this angle is at a minimum (0 degrees) when the sun is directly overhead and is parallel to the zenith
# At sunrise and sunset, the angle is at a maximum (90 degrees) when the sun is on the horizon
# On average in mid-latitude regions, this angle reaches a minimum in the summerimte and maximum in the wintertime
# Let's keep this simple for demonstration purposes since we are modifying canopy, not solar position
theta_list = np.arange(0, 91, 15)  # Solar zenith angles from 0 to 90 degrees in 15 degree increments
print("Solar zenith angles (theta):", theta_list)

# Total incoming shortwave radiation at the top of the canopy
S_toc = 1000  # W/m^2, example value for total incoming shortwave radiation
plt.figure(figsize=(10, 6))
# Calculate the direct radiation under the canopy for each combination of mu, h, and theta
S_b_matrix_list_bymu = []
for mu in (mu_list):
    S_b_matrix = np.zeros((len(h_list), len(theta_list)))
    for i, h in enumerate(h_list):
        for j, theta in enumerate(theta_list):
            S_b_matrix[i, j] = direct_radiation_subcanopy(S_toc, mu, h, theta)
    S_b_matrix_list_bymu.append(S_b_matrix)

    # Plot the results for each combination of mu and h
    markers = ['o', 'd', 's', '^', 'x']  # Different markers for each canopy height
    for i, h in enumerate(h_list):
        plt.plot(theta_list, S_b_matrix[i], marker=markers[i], label=f'Canopy height = {h:.2f} m, mu = {mu:.3f}')
    plt.title(f'Direct Radiation Under Canopy for Varying Solar Zenith Angles with Fixed Extinction Coefficients')
    # plt.title(f'Direct Radiation Under Canopy for Varying Solar Zenith Angles with Fixed Extinction Coefficient {mu}')
    plt.xlabel('Solar zenith angle (degrees)')
    plt.ylabel('Direct Radiation (W/m^2)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
# Plot these by height (five subplots total, one for each height class)
fig, axa = plt.subplots(1, len(h_list), figsize=(15, 5), sharey=True)

for i, h in enumerate(h_list):
    ax = axa[i]
    # plt.figure(figsize=(10, 6))
    for j, mu in enumerate(mu_list):
        ax.plot(theta_list, S_b_matrix_list_bymu[j][i], marker=markers[j], label=f'Extinction Coefficient = {mu:.3f}')
    ax.set_title(f'Canopy Height = {h:.2f} m')
    ax.set_xlabel('Solar zenith angle (degrees)')
    ax.set_ylabel('Direct Radiation (W/m^2)')
    ax.set_ylim(0, 1000)
    ax.grid('lightgray', linestyle='--', linewidth=0.5)
    if i == len(h_list) - 1:
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
# Make a longer mu list
# Calculate the direct radiation under the canopy for each combination of mu, h, and theta
S_b_matrix_list_bymu = []
mu_list = np.arange(0.01, 0.1, 0.02)
for mu in (mu_list):
    S_b_matrix = np.zeros((len(h_list), len(theta_list)))
    for i, h in enumerate(h_list):
        for j, theta in enumerate(theta_list):
            S_b_matrix[i, j] = direct_radiation_subcanopy(S_toc, mu, h, theta)
    S_b_matrix_list_bymu.append(S_b_matrix)

# Plot these by height (five subplots total, one for each height class)
fig, axa = plt.subplots(1, len(h_list), figsize=(15, 5), sharey=True)

for i, h in enumerate(h_list):
    ax = axa[i]
    # plt.figure(figsize=(10, 6))
    for j, mu in enumerate(mu_list):
        ax.plot(theta_list, S_b_matrix_list_bymu[j][i], marker=markers[j], label=f'Extinction Coefficient = {mu:.3f}')
    ax.set_title(f'Canopy Height = {h:.2f} m')
    ax.set_xlabel('Solar zenith angle (degrees)')
    ax.set_ylabel('Direct Radiation (W/m^2)')
    ax.set_ylim(0, 1000)
    ax.grid('lightgray', linestyle='--', linewidth=0.5)
    if i == len(h_list) - 1:
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
# Note that larger SZA are closer to the horizon and thus have less direct radiation under the canopy

In [ ]:
for tau in tau_values:
    # Calculate k from tau
    # k = -np.log(tau)
    print(f'k = -log({tau}) = {-np.log(tau):.2f}')

In [ ]:
for mu in [0, 0.025, 0.033, 0.040, 0.074]:
    print(f'mu = {mu:.3f}')
    for h in h_list:
        # Calculate k as mu * h
        k = mu * h
        print(f'k({h} m) = {k:.2f}')


In [ ]:
# Otto's stuff
# Load topo
new_topo = xr.open_dataset("topo_newveg_prefire.nc")

# Reassign non-positive LAI values to 0.001 (LAI derived from Sentinel imagery)
lai_no0 = xr.where(new_topo["lai"] <= 0, 0.001,new_topo["lai"])  # The makes sure there are no unrealistic lais

# Use empiricial calculation to derive canopy density from LAI
can_den_0 = 0.29*np.log(lai_no0)+0.55                            # Empircal relationship from Pomeroy et al. (2002)

# Threshold canopy density to avoid negative values
can_den_0 = can_den_0.clip(min=0)                                # Removes negative canopy densityies as assigns 0

# Threshold canopy density to avoid extreme values
can_den_1 = xr.where(lai_no0 < 0.15, 0, can_den_0)               # Empirical bounds from Pomeroy et al. (2002)
can_den = xr.where(lai_no0 > 4.72, 1, can_den_1)                 # "    "     "

# Calculate transmissivity from canopy density assuming transmissivity is equivalent to sky view fraction
tau = 1-can_den                                                  # Assumption from Rasmus et al., (2014)

# Compute extinction coefficient (k) from transmissivity and canopy height
# This assumes that
# 1) k here is the "veg_k" and that that is actually mu
# 2) tau_diffuse is equivalent to tau_direct and is also equivalent to sky view fraction
# then, mu = -log(tau_diffuse)/h and this "k" = log(tau_diffuse)/h
k_0 = (-np.log(tau))/(new_topo["veg_height"])                    # Beers law

# This approach assumes tau_diffuse and tau_direct are not equivalent and 
# scales tau_diffuse by 
k_0 = (-0.5*np.log(tau))/(new_topo["veg_height"])

# Threshold k to avoid extreme values
k = xr.where(new_topo["veg_height"] <= 5, 0, k_0)                # avoids extreme values by remove veg heights <5m, will need to experiment with this    
